Import libraries.

In [ ]:
import os
import torch
import shutil
import kagglehub
import numpy as np
from torch import nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix

Setting of the device.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Import dataset.

In [ ]:
path = kagglehub.dataset_download("gpiosenka/sports-classification")

A generated function to extract only given classes

In [ ]:
class SubsetImageFolder(datasets.ImageFolder):
    def __init__(self, root, chosen_classes, transform=None):
        self.chosen_classes = chosen_classes
        super().__init__(root, transform=transform)

    def find_classes(self, directory):
        classes = sorted(self.chosen_classes)
        class_to_idx = {}
        for i in range(len(classes)):
            class_to_idx[classes[i]] = i
        return classes, class_to_idx


CLASSES = ["basketball", "baseball", "tennis", "golf", "boxing", "swimming"]


Preparations of datasets. We will work with gray and then with colored data to compare the impact of color and also to reduce de number of parametters by 3 for the gray images. Our hypothesis is that for sport comparation, color will be usefull so improve accuracy.

In [ ]:
IMG_SIZE = 64

transformGray = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(),
    transforms.ToTensor(),
])

transformColor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_datasetGray = datasets.ImageFolder(os.path.join(path, "train"), transform=transformGray)
val_datasetGray = datasets.ImageFolder(os.path.join(path, "valid"), transform=transformGray)
test_datasetGray = datasets.ImageFolder(os.path.join(path, "test"), transform=transformGray)

train_datasetColor = datasets.ImageFolder(os.path.join(path, "train"), transform=transformColor)
val_datasetColor = datasets.ImageFolder(os.path.join(path, "valid"), transform=transformColor)
test_datasetColor = datasets.ImageFolder(os.path.join(path, "test"), transform=transformColor)

# 6 classes
train_datasetGray6 = SubsetImageFolder(os.path.join(path, "train"), CLASSES, transform=transformGray)
val_datasetGray6 = SubsetImageFolder(os.path.join(path, "valid"), CLASSES, transform=transformGray)
test_datasetGray6 = SubsetImageFolder(os.path.join(path, "test"), CLASSES, transform=transformGray)

train_datasetColor6 = SubsetImageFolder(os.path.join(path, "train"), CLASSES, transform=transformColor)
val_datasetColor6 = SubsetImageFolder(os.path.join(path, "valid"), CLASSES, transform=transformColor)
test_datasetColor6 = SubsetImageFolder(os.path.join(path, "test"), CLASSES, transform=transformColor)

Here is the dataset analysis (i let you do it alvaro) I just printed number of classes and of images for each types of datasets.

In [ ]:
num_classes = len(train_datasetGray.classes)
print("Classes :", num_classes)
print("Train :", len(train_datasetGray), "| Val :", len(val_datasetGray), "| Test :", len(test_datasetGray))

In [ ]:
def convolution():
    # TODO
    pass

Model Net with activation function, regularization and convolution. We set convolution by default to None, if so, it just flat the image.

In [ ]:
class Net(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64,), num_classes=100, convolution = None, activation_function="relu", regularization=None, reg_param=0.0):
        super().__init__()
        self.hidden_sizes = hidden_sizes
        self.num_classes = num_classes
        self.regularization = regularization
        self.reg_param = reg_param

        if convolution is not None:
            self.features = convolution()
        else:
            self.features = nn.Identity()
            
        self.flatten = nn.Flatten()

        self.lst = nn.ModuleList()
        
        self.lst.append(nn.Linear(input_size, hidden_sizes[0]))
        for i in range(1, len(hidden_sizes)):
            self.lst.append(nn.Linear(hidden_sizes[i - 1], hidden_sizes[i]))
        self.lst.append(nn.Linear(hidden_sizes[-1], num_classes))

        if activation_function == "relu":
            self.activation_function = nn.ReLU()
        elif activation_function == "tanh":
            self.activation_function = nn.Tanh()
        elif activation_function == "sigmoid":
            self.activation_function = nn.Sigmoid()

        if regularization == "dropout":
            self.dropouts = nn.ModuleList()
            for i in range(len(hidden_sizes)):
                self.dropouts.append(nn.Dropout(reg_param))

    def forward(self, X):
        X = self.features(X)
        X = self.flatten(X)
        X = self.lst[0](X)
        for i in range(1, len(self.hidden_sizes) + 1):
            X = self.activation_function(X)
            if self.regularization == "dropout":
                X = self.dropouts[i - 1](X)
            X = self.lst[i](X)
        return X

Optimizers and schedulers functions.

In [ ]:
def make_optimizer(name, params, lr, weight_decay=0.0):
    if name == "Adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    elif name == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)


def make_scheduler(name, optimizer):
    if name is None:
        return None
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

Training function.

In [ ]:
def train(model, train_loader, val_loader, epochs=40, lr=1e-4, optimizer_name="Adam", scheduler_name=None, limit_epoch=10):

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    if model.regularization == "weight_decay":
        optimizer = make_optimizer(optimizer_name, model.parameters(), lr, weight_decay=model.reg_param)
    else:
        optimizer = make_optimizer(optimizer_name, model.parameters(), lr)

    scheduler = make_scheduler(scheduler_name, optimizer)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    confusion_matrices = []
    best_val_loss = float('inf')
    epoch_counter = 0

    for epoch in range(epochs):
        
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)

            if model.regularization == "l1":
                l1 = 0
                for p in model.parameters():
                    l1 = l1 + p.abs().sum()
                loss = loss + model.reg_param * l1

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            preds = out.argmax(1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

        train_losses.append(total_loss / total)
        train_accs.append(correct / total)

        
       
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                total_loss += loss.item() * xb.size(0)

                preds = out.argmax(1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(yb.cpu().numpy())

        val_loss = total_loss / total
        val_losses.append(val_loss)
        val_accs.append(correct / total)
        confusion_matrices.append(confusion_matrix(all_labels, all_preds, labels=list(range(model.num_classes))))

        if scheduler is not None:
            scheduler.step()
            
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_losses[-1]:.4f}, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_accs[-1]:.4f}')

        
        if model.regularization == "early_stopping":
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epoch_counter = 0
            else:
                epoch_counter += 1
            if epoch_counter >= limit_epoch:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return {
        "loss_train": train_losses,
        "accuracy_train": train_accs,
        "loss_val": val_losses,
        "accuracy_val": val_accs,
        "confusion_matrices": confusion_matrices,
        "model": model,
    }

Running Function to call with good parametters

In [ ]:
def run_training(train_dataset, val_dataset, num_classes, hidden_sizes=(64,), activation_function="relu", regularization=None, reg_param=0.0,
                 optimizer_name="Adam", scheduler_name=None, epochs=40, lr=1e-4, batch_size=64, limit_epoch=10, seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    first_image, first_label = train_dataset[0]
    input_size = first_image.numel()

    net = Net(input_size, hidden_sizes=hidden_sizes, num_classes=num_classes, activation_function=activation_function, regularization=regularization, reg_param=reg_param)

    metrics = train(net, train_loader, val_loader, epochs=epochs, lr=lr, optimizer_name=optimizer_name, scheduler_name=scheduler_name, limit_epoch=limit_epoch)

    return metrics

Temporary random tests

In [ ]:
def plot_metrics(metrics, cm_epochs=()):
    epochs = range(1, len(metrics["loss_train"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, metrics["accuracy_train"], label="Train")
    axes[0].plot(epochs, metrics["accuracy_val"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(epochs, metrics["loss_train"], label="Train")
    axes[1].plot(epochs, metrics["loss_val"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    for epoch in cm_epochs:
        index = epoch - 1
        if index < 0 or index >= len(metrics["confusion_matrices"]):
            print("No confusion matrix for epoch", epoch)
            continue

        cm = metrics["confusion_matrices"][index]
        plt.figure(figsize=(8, 8))
        plt.imshow(cm, cmap="Blues")
        plt.colorbar()
        plt.title("Confusion matrix - epoch " + str(epoch))
        plt.xlabel("Predicted class")
        plt.ylabel("True class")
        plt.show()

In [ ]:
metrics = run_training(train_datasetGray, val_datasetGray, num_classes=100, hidden_sizes=(256, 128), epochs=10, lr=1e-3)
plot_metrics(metrics, [1, 6, 10])

In [ ]:
metrics = run_training(train_datasetColor6, val_datasetColor6, num_classes=6, hidden_sizes=(256, 128), epochs=10, lr=1e-3)
plot_metrics(metrics, [1, 6, 10])